# 04 — Avaliação da recuperação e do RAG

## Objetivo
Avaliar a recuperação semântica com métricas reprodutíveis, preparar uma fila de candidatos para anotação manual e testar os caminhos de segurança do pipeline.

**Importante:** o conjunto atual em `data/evaluation/retrieval_gold.json` é preliminar. As métricas são diagnósticas, não uma avaliação final, até que os documentos relevantes sejam revisados manualmente e o conjunto seja ampliado.

A avaliação mede o retrieval bruto, antes dos guardrails. O LLM não é necessário para calcular Precision@K, Recall@K e MRR.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
import json
import pandas as pd

from config.settings import settings
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from src.rag.pipeline import RAGPipeline
from src.evaluation.evaluator import evaluate_queries

INDEX_DIR = settings.data_dir / "vectorstore"
GOLD_FILE = ROOT / "data" / "evaluation" / "retrieval_gold.json"

retriever = VectorRetriever.load(INDEX_DIR)
embedder = EmbeddingModel(settings.embedding_model)
pipeline = RAGPipeline(
    embedder=embedder,
    retriever=retriever,
    generator=None,
    top_k=settings.top_k,
    min_relevance_score=settings.min_relevance_score,
    min_scope_score=settings.min_scope_score,
    min_domain_score=settings.min_domain_score,
)

print("Vetores no índice:", retriever.size)
print("Modelo:", settings.embedding_model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vetores no índice: 40950
Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 1. Conjunto de avaliação e status das anotações

O arquivo inicial possui apenas cinco perguntas e parte das anotações foi originada de exemplos previamente exibidos. Por isso, os itens permanecem marcados como **provisórios**.

Não consideramos um documento relevante apenas porque foi recuperado pelo sistema. A confirmação deve ser feita pela leitura do comentário e pelo critério semântico da pergunta.

In [2]:
with GOLD_FILE.open(encoding="utf-8") as file:
    gold_queries = json.load(file)

provisional_count = sum(
    item.get("annotation_status", "provisional") != "reviewed"
    for item in gold_queries
)
print("Perguntas no conjunto:", len(gold_queries))
print("Perguntas ainda não revisadas:", provisional_count)
if provisional_count:
    print("ATENÇÃO: métricas preliminares; revise as anotações antes de tratá-las como resultado final.")

for item in gold_queries:
    print(f"- {item['question']} | IDs relevantes: {len(item['relevant_ids'])}")

Perguntas no conjunto: 5
Perguntas ainda não revisadas: 5
ATENÇÃO: métricas preliminares; revise as anotações antes de tratá-las como resultado final.
- Quais são os principais problemas relacionados à entrega? | IDs relevantes: 5
- Quais problemas aparecem nos relatos de atraso ou demora na entrega? | IDs relevantes: 5
- Quais reclamações aparecem nas avaliações de baixa nota? | IDs relevantes: 2
- O que os clientes relatam sobre a qualidade dos produtos? | IDs relevantes: 5
- Quais problemas de entrega são mencionados em relação aos Correios? | IDs relevantes: 2


## 2. Fila ampliada para anotação manual

Para ampliar a avaliação sem inventar rótulos, este bloco recupera até 20 candidatos por pergunta em dez temas. O CSV deve ser revisado manualmente: preencha `relevance_manual` com `sim` ou `não` e registre uma justificativa curta.

A fila é um apoio à anotação, não um novo conjunto-ouro automático. Não use os scores como rótulo de relevância.

In [3]:
annotation_questions = [
    "Quais problemas os clientes relatam sobre atraso na entrega?",
    "Quais reclamações aparecem sobre embalagem e estado do produto recebido?",
    "O que os clientes dizem sobre a qualidade dos produtos?",
    "Quais relatos mencionam produto diferente ou pedido incompleto?",
    "Quais problemas os clientes relatam sobre reembolso ou devolução?",
    "Como os clientes avaliam o atendimento e a comunicação com vendedores?",
    "Quais aspectos positivos aparecem nas avaliações dos clientes?",
    "Quais reclamações aparecem em avaliações com nota 1 ou 2?",
    "O que os clientes relatam sobre os Correios e a transportadora?",
    "Quais problemas aparecem relacionados ao prazo prometido e à entrega efetiva?",
]

annotation_rows = []
query_vectors = embedder.encode(annotation_questions)
for question, vector in zip(annotation_questions, query_vectors):
    candidates = retriever.search(
        vector,
        top_k=20,
        min_score=settings.min_relevance_score,
    )
    for rank, item in enumerate(candidates, start=1):
        annotation_rows.append({
            "question": question,
            "rank": rank,
            "document_id": item["document_id"],
            "score": round(float(item["score"]), 6),
            "review_score": item.get("metadata", {}).get("review_score"),
            "review_creation_date": item.get("metadata", {}).get("review_creation_date"),
            "comment": item["text"],
            "relevance_manual": "",
            "annotation_note": "",
        })

annotation_df = pd.DataFrame(annotation_rows)
annotation_file = ROOT / "data" / "evaluation" / "retrieval_annotation_candidates.csv"
annotation_file.parent.mkdir(parents=True, exist_ok=True)
annotation_df.to_csv(annotation_file, index=False, encoding="utf-8-sig")
print("Perguntas para anotação:", len(annotation_questions))
print("Candidatos exportados:", len(annotation_df))
print("Arquivo:", annotation_file.relative_to(ROOT))
display(annotation_df.head(10))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Perguntas para anotação: 10
Candidatos exportados: 200
Arquivo: data\evaluation\retrieval_annotation_candidates.csv


,question,rank,document_id,score,review_score,review_creation_date,comment,relevance_manual,annotation_note
0,Quais problemas os clientes relatam sobre atra...,1,dd6e0f7168d75cd78769333f7630a036,0.825602,3,2018-04-04T00:00:00,O atraso na entrega prejudicou a obra,,
1,Quais problemas os clientes relatam sobre atra...,2,14ab06af2677555bfcf9ff2e64a1b203,0.815290,1,2018-06-12T00:00:00,Sem satisfação sobre atraso na entrega,,
2,Quais problemas os clientes relatam sobre atra...,3,a553875413fd30a7f78cd701b0064a6f,0.811960,5,2018-03-04T00:00:00,"Recomendo o produto e o fornecedor, pois o atr...",,
3,Quais problemas os clientes relatam sobre atra...,4,f02d4978f58368f3346d8450d52f0ba2,0.788850,1,2018-02-21T00:00:00,Pessimo muita demora na entrega,,
4,Quais problemas os clientes relatam sobre atra...,5,1c99ea0e3859fceac16388d107c7caa4,0.783570,1,2018-06-28T00:00:00,demora na entrega,,
5,Quais problemas os clientes relatam sobre atra...,6,0fc42b14539f4177f534382d69d82d3c,0.783570,1,2018-06-09T00:00:00,demora na entrega,,
6,Quais problemas os clientes relatam sobre atra...,7,ab796da379bad664fa884f34711b0ca8,0.780481,5,2017-05-13T00:00:00,Muita demora na entrega,,
7,Quais problemas os clientes relatam sobre atra...,8,95e7f613e3dfe80a79306a8c80084739,0.780481,1,2018-03-08T00:00:00,Muita demora na entrega,,
8,Quais problemas os clientes relatam sobre atra...,9,3ffed17f5010b9c6d33706b790732482,0.780481,1,2017-08-16T00:00:00,Muita demora na entrega,,
9,Quais problemas os clientes relatam sobre atra...,10,72926e5236768ffc912764ca429d0e00,0.780481,1,2018-03-09T00:00:00,Muita demora na entrega,,


## 3. Métricas de recuperação

- **Precision@K:** quantidade de relevantes recuperados dividida por K (mesmo quando retornam menos de K itens).
- **Recall@K:** quantidade de relevantes recuperados dividida pelo total de relevantes anotados.
- **MRR:** inverso da posição do primeiro documento relevante; zero quando não há acerto.

As métricas são calculadas pela implementação reutilizada em `src/evaluation/evaluator.py`.

In [4]:
rows = evaluate_queries(pipeline, gold_queries, k=settings.top_k)
metrics = pd.DataFrame(rows)

display(metrics[["question", "precision_at_k", "recall_at_k", "mrr"]])

summary = {
    f"Precision@{settings.top_k}": float(metrics["precision_at_k"].mean()),
    f"Recall@{settings.top_k}": float(metrics["recall_at_k"].mean()),
    "MRR": float(metrics["mrr"].mean()),
    "queries": int(len(metrics)),
    "annotation_status": "provisional" if provisional_count else "reviewed",
}
display(pd.DataFrame([summary]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,question,precision_at_k,recall_at_k,mrr
0,Quais são os principais problemas relacionados...,1.0,1.0,1.0
1,Quais problemas aparecem nos relatos de atraso...,0.0,0.0,0.0
2,Quais reclamações aparecem nas avaliações de b...,0.4,1.0,1.0
3,O que os clientes relatam sobre a qualidade do...,0.0,0.0,0.0
4,Quais problemas de entrega são mencionados em ...,0.0,0.0,0.0


,Precision@5,Recall@5,MRR,queries,annotation_status
0,0.28,0.4,0.4,5,provisional


## 4. Auditoria qualitativa

Compare os comentários recuperados com os critérios da pergunta. Uma métrica zero pode indicar falha de recuperação ou uma anotação incompleta; não altere o rótulo sem ler o texto.

O conjunto-ouro deve incluir exemplos relevantes encontrados por amostragem independente, e não somente os primeiros resultados da busca avaliada.

In [5]:
inspection_question = "Quais são os principais problemas relacionados à entrega?"
candidates = pipeline.retrieve_candidates(inspection_question)

for item in candidates:
    print(f"\n[{item['document_id']}] score={item['score']:.4f}")
    print("Nota:", item.get("metadata", {}).get("review_score"))
    print(item["text"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


[7d8d252cc0ae002b13db033ec1cd9dcb] score=0.7659
Nota: 1
Demora de entrega

[048f89d5550f6d495dbfb726b23ba2a1] score=0.7516
Nota: 2
Atraso na entrega

[a1074ef6728469383f2a8bf58fd98e51] score=0.7516
Nota: 1
Atraso na entrega

[db6e80117789aa15d03c3406c7429845] score=0.7446
Nota: 1
Demora na entrega

[b02ae085148e5e1eb6b11142ea9c653c] score=0.7446
Nota: 2
Demora na entrega


## 5. Teste de insuficiência de evidências

São dois testes controlados: (1) limiar de relevância alto para simular ausência de candidatos; (2) limiar de suficiência alto para simular candidatos fracos. Esses testes verificam os caminhos do código, mas não substituem a calibração dos limiares em um conjunto de validação.

In [6]:
no_evidence_pipeline = RAGPipeline(
    embedder=embedder,
    retriever=retriever,
    generator=None,
    top_k=settings.top_k,
    min_relevance_score=0.99,
    min_scope_score=settings.min_scope_score,
    min_domain_score=settings.min_domain_score,
)
no_evidence_result = no_evidence_pipeline.ask(inspection_question)
print("Status sem candidatos:", no_evidence_result["evidence_status"])
print("Resposta:", no_evidence_result["answer"])
assert no_evidence_result["evidence_status"] == "insufficient_evidence"
assert no_evidence_result["evidence"] == []

weak_evidence_pipeline = RAGPipeline(
    embedder=embedder,
    retriever=retriever,
    generator=None,
    top_k=settings.top_k,
    min_relevance_score=settings.min_relevance_score,
    min_scope_score=0.99,
    min_domain_score=settings.min_domain_score,
)
weak_result = weak_evidence_pipeline.ask(inspection_question)
print("\nStatus com candidatos fracos:", weak_result["evidence_status"])
assert weak_result["evidence_status"] == "insufficient_evidence"
assert weak_result["evidence"] == []

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Status sem candidatos: insufficient_evidence
Resposta: Não há evidências suficientes na base de conhecimento para responder a esta pergunta.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Status com candidatos fracos: insufficient_evidence


## 6. Teste de pergunta fora do domínio

O pipeline deve identificar uma pergunta sem relação com avaliações da Olist e não encaminhar evidências ao gerador.

In [7]:
out_of_domain_question = "Qual é a previsão do tempo para amanhã?"
out_of_domain_result = pipeline.ask(out_of_domain_question)
print("Pergunta:", out_of_domain_question)
print("Score de domínio:", round(out_of_domain_result["domain_score"], 4))
print("Limite configurado:", settings.min_domain_score)
print("Status:", out_of_domain_result["evidence_status"])
print("Resposta:", out_of_domain_result["answer"])
assert out_of_domain_result["evidence_status"] == "out_of_scope"
assert out_of_domain_result["evidence"] == []

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Pergunta: Qual é a previsão do tempo para amanhã?
Score de domínio: 0.0654
Limite configurado: 0.45
Status: out_of_scope
Resposta: A pergunta parece estar fora do escopo da base de conhecimento Olist. Não é possível responder com segurança usando as avaliações disponíveis.


## 7. Exportação dos resultados

O JSON registra as métricas e o status das anotações. Enquanto houver itens provisórios, os resultados devem ser apresentados como diagnóstico inicial, não como avaliação final do desempenho do RAG.

In [8]:
output_file = ROOT / "data" / "evaluation" / "retrieval_results_notebook.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
output_file.write_text(
    json.dumps({"summary": summary, "details": rows}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Resultados salvos em:", output_file.relative_to(ROOT))
print("Status das anotações:", summary["annotation_status"])

Resultados salvos em: data\evaluation\retrieval_results_notebook.json
Status das anotações: provisional


## 8. Conclusões e próximos passos

1. Revisar manualmente o CSV `retrieval_annotation_candidates.csv`, preenchendo `relevance_manual` e `annotation_note`.
2. Atualizar `retrieval_gold.json` somente com IDs cuja relevância tenha sido confirmada pela leitura do comentário.
3. Marcar uma pergunta como `annotation_status: "reviewed"` somente após revisão independente.
4. Reexecutar este notebook e o script `scripts/evaluate_retrieval.py`.
5. Comparar as métricas por pergunta e investigar erros antes de ajustar o modelo ou os limiares.

Não ajuste rótulos apenas para elevar as métricas. A qualidade do conjunto-ouro é parte central da avaliação.

**Notebook integrado:** [00 — RAG completo](00_rag_completo.ipynb).